# Runnable 인터페이스 기초

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `RunnablePassthrough`로 입력을 그대로 전달하거나 `.assign()`으로 새 키를 추가할 수 있어요
2. `RunnableParallel`로 여러 체인을 동시에 실행하고 결과를 딕셔너리로 받을 수 있어요
3. `RunnableLambda`로 Python 함수를 체인에 삽입해 전처리·후처리를 구현할 수 있어요
4. 세 가지 Runnable을 조합해 실무형 마케팅 파이프라인을 구성할 수 있어요

## 사전 지식

- `02_Chain.ipynb`의 파이프 연산자와 `PromptTemplate` 사용법
- `03_LCEL_basic.ipynb`의 `invoke`, `batch`, `stream` 메서드
- Python 딕셔너리, 람다 표현식

## 이전 노트북 연결

`03_LCEL_basic.ipynb`에서 체인을 다양한 방식으로 실행하는 방법을 배웠어요. 이번에는 체인 내부 데이터 흐름을 세밀하게 제어하는 세 가지 핵심 Runnable을 살펴볼게요.

아래 다이어그램은 세 Runnable의 역할과 상호관계를 보여줘요.

```mermaid
flowchart TD
    IN["원본 입력<br/>{product_name, price}"]:::input

    IN --> PT["RunnablePassthrough<br/>입력 그대로 통과"]:::process
    IN --> PA["RunnablePassthrough.assign<br/>새 키 추가<br/>discount_rate, current_time"]:::process
    PA --> RP["RunnableParallel<br/>병렬 실행"]:::process
    RP --> C1["체인 1<br/>상세설명 LLM"]:::process
    RP --> C2["체인 2<br/>한줄요약 LLM"]:::process
    RP --> C3["체인 3<br/>알림톡문구 LLM"]:::process
    C1 --> RL["RunnableLambda<br/>결과 합치기"]:::process
    C2 --> RL
    C3 --> RL
    RL --> OUT["최종 출력<br/>마케팅 패키지"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

Runnable 인터페이스(Runnable Interface)는 LangChain의 핵심 추상화 개념이에요. 모든 LangChain 컴포넌트(프롬프트, 모델, 출력 파서 등)가 이 인터페이스를 구현해요.

주요 메서드:
- `invoke()`: 단일 입력을 동기 실행
- `batch()`: 여러 입력을 배치 처리
- `stream()`: 출력을 스트리밍 방식으로 반환
- `ainvoke()`, `abatch()`, `astream()`: 비동기 버전

## 1. RunnablePassthrough

`RunnablePassthrough(러너블 패스스루)`는 입력을 변경하지 않고 그대로 전달하거나, `.assign()`으로 새 키를 추가해 전달할 수 있는 Runnable이에요.

### 왜 필요한가요?

LCEL 파이프라인에서 데이터는 한 방향으로만 흘러요. 그런데 실제로는 **원본 입력을 보존하면서 LLM 호출 결과도 함께 전달**해야 하는 경우가 많아요. `RunnablePassthrough`가 이 문제를 해결해요.

### 주요 사용법

| 사용법 | 동작 |
|--------|------|
| `RunnablePassthrough()` | 입력을 변형 없이 그대로 통과시켜요 (항등 함수처럼 동작해요) |
| `RunnablePassthrough.assign(key=fn)` | 원본 입력을 유지하면서 새로운 키-값 쌍을 동적으로 추가해요 |

> **비유**: `RunnablePassthrough`는 컨베이어 벨트 위의 물건을 다음 단계로 그대로 보내는 역할이에요. `.assign()`은 물건 위에 라벨을 붙여서 보내는 역할이에요.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

### 1.1 RunnablePassthrough() - 입력 그대로 전달

가장 기본적인 사용법이에요. 입력 딕셔너리가 변형 없이 그대로 반환돼요. 파이프라인 분기점에서 원본 데이터를 유지할 때 사용해요.

In [2]:
# ---------------------------------------------------
# RunnablePassthrough(): 입력을 변형 없이 그대로 통과시키기
# ---------------------------------------------------
# RunnablePassthrough: 항등 함수처럼 동작해요 — 입력 = 출력
# 파이프라인 분기점에서 원본 데이터를 유지할 때 사용해요

passthrough = RunnablePassthrough()
res = passthrough.invoke({"question": "안녕하세요", "user": "elixir"})
print(f' ==> [Line 9]: \033[38;2;21;116;198m[res]\033[0m({type(res).__name__}) = \033[38;2;200;46;125m{res}\033[0m')


 ==> [Line 9]: [res](dict) = {'question': '안녕하세요', 'user': 'elixir'}


### 1.2 RunnablePassthrough.assign() - 새 키 추가

`.assign()`은 원본 입력의 모든 키를 유지하면서 새로운 키-값 쌍을 동적으로 추가해요.

**언제 사용하면 좋을까요?**
- 체인 중간에서 원본 데이터를 보존하면서 **파생 값이나 메타데이터를 추가**할 때
- 입력 데이터를 기반으로 **동적으로 새로운 정보를 계산**할 때 (예: 할인율 계산, 타임스탬프 추가)
- 여러 단계의 체인에서 **이전 단계의 모든 정보를 유지**하면서 점진적으로 데이터를 확장할 때

In [13]:
# ---------------------------------------------------
# RunnablePassthrough.assign()으로 원본 입력에 파생 값 추가하기
# ---------------------------------------------------
# ============================================================
# TODO: assign()에 새 키를 추가해 체인을 실행해봐요
# 힌트: lambda x: 형태로 원본 입력 딕셔너리(x)에서 값을 꺼내 계산해요
# 예상 결과: product_name, price, discount_rate, current_time이 모두 포함된 응답이 나와요
# ============================================================

from datetime import datetime

# 1단계: 프롬프트 템플릿 정의
# 입력할 값: 상품명, 가격, 세일 여부
product_prompt = PromptTemplate.from_template(
  """
  다음 상품에 매력적인 설명을 작성해주세요:

  사용자명: {username}님
  상품명:   {product_name}
  정가:     {price}원
  할인율:   {discount_rate}%
  현재시간: {current_time}
  판매여부: {is_on_sale}

  => 위 정보를 바탕으로 고객에게 어필할 수 있는 설명을 작성해주세요. 비회원인 경우 회원 가입 유도 문구를 포함해주세요.
  """
)

# 2단계: RunnablePassthrough.assign()

rpa = RunnablePassthrough.assign(
  discount_rate = lambda x : 30 if x.get("is_on_sale") else 0,
  current_time = lambda x : datetime.now().strftime("%Y년, %m월 %d일 %H시 %M분"),
  username = lambda x : x.get("username", None)
)

chain = (
  rpa
  | product_prompt
  | model
  | StrOutputParser()
)

# res = chain.invoke({
#   "username": "김철수",
#   "product_name": "영양제",
#   "price": "100",
#   "discount_rate": "10",
#   "current_time": "10"
# })

res = chain.invoke({
  "username": "Elixir",
  "product_name": "무선 블루투스 이어폰",
  "price": 100_000,
  "is_on_sale": True
})

print(f' ==> [Line 58]: \033[38;2;218;126;184m[res]\033[0m({type(res).__name__}) = \033[38;2;127;0;125m{res}\033[0m')


 ==> [Line 58]: [res](str) = 🎧 무선 블루투스 이어폰 - 당신의 음악 경험을 혁신하다! 🎧

안녕하세요, Elixir님! 음악을 사랑하는 당신을 위한 완벽한 선택, 무선 블루투스 이어폰이 30% 할인된 특별가로 준비되었습니다. 정가는 100,000원이지만, 지금 이 시점인 2026년 3월 23일에만 특별히 70,000원에 만나보실 수 있습니다!

**주요 Features:**
- **무선 편리함:** 복잡한 선에서 해방되어 자유롭게 움직일 수 있습니다. 운동, 출퇴근, 여가 시간에 모두 함께하세요!
- **탁월한 사운드 품질:** 깊고 풍부한 저음과 선명한 고음을 통해 음악을 생생하게 감상할 수 있습니다.
- **최상의 착용감:** 인체 공학적 디자인으로 어떤 환경에서도 편안하게 착용할 수 있습니다.
- **긴 배터리 수명:** 한 번의 충전으로 긴 시간 동안 즐길 수 있어, 하루 종일 음악과 함께하실 수 있습니다.

지금 바로 다가오는 새로운 사운드의 세계를 경험해보세요! 할인된 가격은 한정된 시간 동안만 제공되니 절대 놓치지 마세요!

🌟 **비회원 혜택:** 회원 가입 시 추가 할인 및 다양한 이벤트 소식을 받아보실 수 있습니다. 지금 가입하시면 더 많은 혜택이 기다리고 있습니다!

이 무선 블루투스 이어폰과 함께 당신의 하루를 더욱 특별하게 만들어보세요. 클릭 한 번으로 쇼핑몰로 이동하세요!  ⬇️

[쇼핑하기]


## 2. RunnableParallel

`RunnablePassthrough`로 입력을 제어하는 방법을 알아봤어요. 이제 여러 체인을 동시에 실행하는 `RunnableParallel(러너블 패러렐)`을 살펴볼게요.

`RunnableParallel`은 딕셔너리 형태로 여러 Runnable을 정의하고, 동일한 입력을 각 Runnable에 동시에 전달해요. 결과는 키-값 쌍의 딕셔너리로 반환돼요.

### 주요 특징
- 여러 LLM 호출을 동시에 실행해 총 응답 시간을 줄여요
- 각 Runnable의 결과를 키로 구분해 딕셔너리로 반환해요
- 같은 입력으로 다양한 관점의 응답을 한 번에 생성할 때 유용해요

In [17]:
from langchain_core.runnables import RunnableParallel

# ---------------------------------------------------
# RunnableParallel: 같은 입력으로 여러 체인을 동시에 실행하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 parallel_chain을 실행하고 "장점", "단점", "활용분야" 키를 확인해요
# 힌트: parallel_chain.invoke({"topic": "원하는 주제"})
# 예상 결과: {"장점": "...", "단점": "...", "활용분야": "..."} 딕셔너리가 반환돼요
# ============================================================

positive_prompt = PromptTemplate.from_template(
"""
{topic}의 장점을 3줄로 나열해줘.
"""
)

negative_prompt = PromptTemplate.from_template(
"""
{topic}의 단점을 3줄로 나열해줘.
"""
)

util_prompt = PromptTemplate.from_template(
"""
{topic}의 주요 활용분야를 3줄로 나열해줘.
"""
)

rp = RunnableParallel({
  "positive": positive_prompt | model | StrOutputParser(),
  "negative": negative_prompt | model | StrOutputParser(),
  "util": util_prompt | model | StrOutputParser()
})

chain = rp

res = chain.invoke({"topic": "블록체인"})
print(f' ==> [Line 34]: \033[38;2;97;203;31m[res]\033[0m({type(res).__name__}) = \033[38;2;227;32;147m{res}\033[0m')


 ==> [Line 34]: [res](dict) = {'positive': '1. **투명성**: 블록체인은 모든 거래 기록이 공개되어 있어 사용자들이 쉽게 검증할 수 있어 신뢰를 높입니다.  \n2. **보안성**: 분산된 네트워크 구조로 인해 해킹이나 데이터 위조가 어려워 높은 수준의 보안을 제공합니다.  \n3. **탈중앙화**: 중앙 권한 없이 거래가 이루어져 사용자 간의 직접적인 상호작용이 가능하며, 권력 집중을 방지합니다.  ', 'negative': '1. **확장성 문제**: 많은 사용자와 거래가 동시에 발생할 경우, 처리 속도가 느려지고 수수료가 증가할 수 있습니다.  \n2. **에너지 소비**: 특히 작업 증명(Proof of Work) 기반의 블록체인은 높은 에너지를 소모하여 환경에 부담을 줄 수 있습니다.  \n3. **규제 및 법적 이슈**: 블록체인의 익명성과 분산성으로 인해 법적인 규제가 어렵고, 사용자 보호가 취약할 수 있습니다.  ', 'util': '1. **금융 및 결제 시스템**: 블록체인은 안전하고 투명한 거래를 가능하게 하여 국제 송금 및 결제 과정의 효율성을 높입니다.  \n2. **스마트 계약**: 자동 실행되는 계약을 통해 중개 없이 다양한 거래를 처리할 수 있어 시간과 비용을 절감합니다.  \n3. **공급망 관리**: 제품의 출처와 이동 과정을 추적 가능하게 하여 신뢰성과 투명성을 높이고 위조를 방지합니다.  '}


### 병렬 실행의 동작 방식 이해하기

`RunnableParallel`은 여러 체인을 병렬로 실행하지만, `invoke()` 메서드는 모든 체인이 완료될 때까지 기다려요. 즉, 사용자는 가장 느린 체인이 끝날 때까지 대기해요.

**데이터 흐름**:
```
입력 {"topic": "..."} ──┬── chain1 (장점) ──┐
                        ├── chain2 (단점) ──┼── 결과 {"장점": ..., "단점": ..., "활용분야": ...}
                        └── chain3 (활용) ──┘
```

다음 셀에서 순차 실행, 병렬 실행, 비동기 실행의 시간을 직접 비교해볼게요.

In [ ]:
import time

# ---------------------------------------------------
# 순차 실행 vs 병렬 실행 vs 비동기 실행 시간 비교하기
# ---------------------------------------------------


In [ ]:
from datetime import datetime

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# ---------------------------------------------------
# 실전 예제: Passthrough + Parallel + Lambda 조합
# 하나의 상품 입력으로 여러 채널용 마케팅 카피를 동시에 생성하기
# ---------------------------------------------------


### 실전 응용: Passthrough + Parallel + Lambda 조합

세 가지 Runnable을 조합해 **하나의 입력으로 여러 형태의 결과물을 동시에 생성**하는 파이프라인을 만들어볼게요.

위 코드 셀의 파이프라인 구조를 정리하면 다음과 같아요:

```
원본 입력 → Passthrough.assign() → RunnableParallel → Lambda(합치기)
            (할인율/시간 추가)      ├─ 상세설명 (LLM)
                                   ├─ 한줄요약 (LLM)
                                   ├─ 알림톡문구 (LLM)
                                   └─ 원본입력 (Passthrough)
```

이 패턴은 실무에서 자주 활용돼요:
- 같은 데이터로 다양한 채널용 콘텐츠를 동시에 생성할 때 (웹, 앱 푸시, 문자)
- A/B 테스트용 여러 버전의 카피를 한 번에 생성할 때
- 분석 결과를 여러 관점에서 동시에 요약할 때

## 3. RunnableLambda

`RunnableParallel`로 병렬 실행을 다뤄봤어요. 이번에는 일반 Python 함수를 체인에 삽입하는 `RunnableLambda(러너블 람다)`를 살펴볼게요.

`RunnableLambda`는 일반 Python 함수를 Runnable로 변환해 파이프라인에 포함시켜요. 체인 중간에 커스텀 로직을 삽입할 수 있어요.

### 주요 활용 사례
- 입력 전처리: 사용자 친화적 형식을 LLM 입력 형식으로 변환해요
- 출력 후처리: LLM 응답에서 특정 부분만 추출하거나 포맷팅해요
- 데이터 타입 변환: 문자열 → 딕셔너리, 딕셔너리 → 문자열 등의 변환이 필요할 때 사용해요

In [27]:
from langchain_core.runnables import RunnableLambda

# ---------------------------------------------------
# RunnableLambda: 일반 Python 함수를 파이프라인에 삽입하기
# ---------------------------------------------------

def add_prefix(text):
    return f"≠♡♡♡♡♡♡♡♡♡♡♡♡♡ {text}"

prompt = PromptTemplate.from_template("{question}에 대해 간단히 설명해줘")

chain = (
    prompt
    | model
    | StrOutputParser()
    | RunnableLambda(add_prefix)
)

res = chain.invoke({"question": "블록체인"})
print(f' ==> [Line 19]: \033[38;2;14;24;3m[res]\033[0m({type(res).__name__}) = \033[38;2;252;182;48m{res}\033[0m')


 ==> [Line 19]: [res](str) = ≠♡♡♡♡♡♡♡♡♡♡♡♡♡ 블록체인은 데이터를 안전하게 저장하고 공유할 수 있는 분산형 데이터베이스 기술입니다. 이 기술은 여러 개의 '블록'이 체인 형태로 연결되어 정보를 기록하는 구조로 되어 있습니다. 각 블록에는 거래 기록, 타임스탬프, 해시값(이전 블록의 정보와 연결하는 암호화된 값) 등이 포함되어 있습니다.

블록체인의 주요 특징은 다음과 같습니다:

1. **탈중앙화**: 중앙 권한 없이 여러 사용자가 데이터를 검증하고 저장할 수 있어, 시스템의 신뢰성을 높입니다.
2. **투명성**: 모든 거래가 공개적으로 기록되며, 누구나 이를 확인할 수 있습니다.
3. **변경 불가능성**: 블록체인에 기록된 데이터는 변경하거나 삭제할 수 없어, 데이터의 무결성을 보장합니다.
4. **보안성**: 암호화 기술을 사용하여 데이터의 안전성을 강화합니다.

이러한 특성 덕분에 블록체인은 금융 거래, 스마트 계약, 공급망 관리 등 다양한 분야에서 활용되고 있습니다.


In [33]:
# ---------------------------------------------------
# 여러 RunnableLambda를 순서대로 연결하기 (데이터 타입 변환 포함)
# ---------------------------------------------------

# RunnableLambda는 감싼 함수의 인자는 항상 이전 단계의 전체 출력을 받는다.
def extract_keywords(text):
    """미리 정의된 키워드 목록과 매칭해 추출하기"""
    keywords = ["파이썬", "머신러닝", "딥러닝", "데이터", "AI", "알고리즘"]
    found = [kw for kw in keywords if kw in text]
    # 반환 타입이 문자열 → 딕셔너리로 바뀜 (다음 단계에서 딕셔너리로 받음)
    return {"text": text, "keywords": found, "keyword_count": len(found)}

def format_output(data):
    """딕셔너리를 사람이 읽기 좋은 형태로 포맷팅 — 딕셔너리 → 문자열 변환"""
    return f"""
        원본 텍스트: {data['text'][:100]}...

        추출된 키워드: {', '.join(data['keywords']) if data['keywords'] else '없음'}
        키워드 개수: {data['keyword_count']}
    """

prompt = PromptTemplate.from_template("{topic}에 대해 설명해줘")
chain = (
    prompt | model | StrOutputParser() | RunnableLambda(extract_keywords) | RunnableLambda(format_output)
)

res = chain.invoke({"topic": "팝캣티움"})
print(f' ==> [Line 27]: \033[38;2;74;30;63m[res]\033[0m({type(res).__name__}) = \033[38;2;105;165;55m{res}\033[0m')


 ==> [Line 27]: [res](str) = 
        원본 텍스트: "팝캣티움"은 한국의 인기 있는 인터넷 밈 및 캐릭터이자, 특히 소셜 미디어와 커뮤니티에서 인기를 끌었던 캐릭터입니다. 이 캐릭터는 주로 귀엽고 앙증맞은 고양이의 모습으로 표현되며...

        추출된 키워드: 없음
        키워드 개수: 0
    


In [34]:
# ---------------------------------------------------
# RunnableLambda를 입력 전처리·출력 후처리 양쪽에 사용하기
# ---------------------------------------------------
# ============================================================
# TODO: language를 "영어"로 바꿔서 실행하고 응답이 달라지는지 확인해요
# 힌트: invoke()의 딕셔너리에서 "language" 값을 변경해요
# 예상 결과: prepare_input()이 "{language}로 {topic}에 대해 설명해줘" 문자열을 생성해요
# ============================================================

def prepare_input(data):
    topic = data.get("topic", "")
    language = data.get("language", "한국어")
    return {
        "question": f"{language}로 {topic}에 대해 설명해줘."
    }

def process_output(text):
    return f"[AI 응답이올시다] {text}"

chain = (
    RunnableLambda(prepare_input) 
    | PromptTemplate.from_template("{question}")
    | model
    | StrOutputParser()
    | RunnableLambda(process_output) 
)

res = chain.invoke({"topic": "개발", "language": "english"})
print(f' ==> [Line 28]: \033[38;2;247;125;28m[res]\033[0m({type(res).__name__}) = \033[38;2;92;80;195m{res}\033[0m')


 ==> [Line 28]: [res](str) = [AI 응답이올시다] Sure! Development refers to the process of creating and improving products, services, or systems through the application of various skills and methodologies. It encompasses a wide range of fields, including software development, web development, game development, and mobile app development. Here’s a brief overview of these key areas:

1. **Software Development**: This involves designing, programming, testing, and maintaining software applications. Developers typically use programming languages such as Python, Java, C++, and others to create software that meets users' needs.

2. **Web Development**: This focuses on building websites and web applications. It can be divided into two main areas: front-end development (the visual part of a website that users interact with) and back-end development (the server-side logic and database management). Technologies used include HTML, CSS, JavaScript for front-end and server-side languages like Node.js, Ruby

## 4. 종합 예제: 세 가지 Runnable 조합하기

`RunnablePassthrough`, `RunnableParallel`, `RunnableLambda`를 모두 배웠어요. 이번에는 세 가지를 함께 조합해 실무형 마케팅 패키지 생성 파이프라인을 만들어볼게요.

이 예제는 앞서 섹션 2에서 다룬 실전 응용과 동일한 구조이지만, 최종 출력을 딕셔너리 대신 사람이 바로 읽을 수 있는 보고서 형태의 문자열로 반환해요.

In [21]:
from datetime import datetime

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# ---------------------------------------------------
# 종합 예제: 세 가지 Runnable 조합 (보고서 형태 출력)
# ---------------------------------------------------

# 1. 상세설명, 한줄설명, 알림톡 문구
# 상세설명: 사용자명, 상품명, 정가, 할인율, 현재 시간
# 한줄설명: 상품명, 가격, 할인율
# 알림톡 문구: 상품명, 가격, 할인율, 현재 시간, 사용자명
detail_prompt = PromptTemplate.from_template(
    "다음 상품에 대한 매력적인 설명을 작성해주세요:\n"
    "사용자명: {user_name}\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "현재 시간: {current_time}\n"
    "→ 위 정보를 바탕으로 고객에게 어필할 수 있는 상세 설명을 작성해주세요. "
    "비회원(사용자명이 없거나 None인 경우)인 경우 회원 가입 유도 문구를 포함해주세요."
)

# 한 줄 요약 — 배너/광고 카피용
summary_prompt = PromptTemplate.from_template(
    "다음 상품 정보를 기반으로, 마케팅용 한 줄 카피만 1줄로 작성해주세요.\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "→ 말 줄임표 없이 완전한 문장으로."
)

# 알림톡 문구 — 카카오 알림톡/문자 발송용
sms_prompt = PromptTemplate.from_template(
    "아래 정보를 이용해, 카카오 알림톡 형식의 짧은 문구를 작성해주세요.\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "현재 시간: {current_time}\n"
    "사용자명: {user_name}\n"
    "조건:\n"
    "- 회원이면 이름을 직접 불러주는 느낌으로 시작\n"
    "- 비회원이면 '고객님'으로 부르고, 마지막에 회원가입 유도 문구 1줄 추가\n"
)

# 2단계 RunnablePassthrough.assign 사용
enrich_chain = RunnablePassthrough.assign(
    discount_rate=lambda x : 30 if x.get('is_on_sale') else 0,
    current_time=lambda x : datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분"),
    user_name=lambda x : x.get("user_name", None)
)

# 3단계 RunnableParallel - enrich 된 4개의 작업을 병렬 실행
parallel_chain = RunnableParallel(
    original=RunnablePassthrough(),
    detail=detail_prompt | model | StrOutputParser(),
    summary=summary_prompt | model | StrOutputParser(),
    sms=sms_prompt | model | StrOutputParser(),
)

# 4 단계: RunnableLambda - 병렬처리 결과를 하나의 딕셔너리로 합치기

def merge_parallel_outputs(data):
    original = data['original']
    return {
        **original,
        "상세설명": data['detail'],
        "한줄요약": data['summary'],
        "알림톡": data['sms']
    }

# agg
merge_chain = RunnableLambda(merge_parallel_outputs)

# 5단계: 최종 파이프라인
chain = (
    enrich_chain
    | parallel_chain
    | merge_chain
)

res = chain.invoke({
    "product_name": "무선 블루투스 이어폰",
    "price": 100_000,
    "is_on_sale": True
})

print(f' ==> [Line 97]: \033[38;2;168;0;137m[res]\033[0m({type(res).__name__}) = \033[38;2;238;212;33m{res}\033[0m')

res



 ==> [Line 97]: [res](dict) = {'product_name': '무선 블루투스 이어폰', 'price': 100000, 'is_on_sale': True, 'discount_rate': 30, 'current_time': '2026년 03월 23일 16시 14분', 'user_name': None, '상세설명': '**🎧 무선 블루투스 이어폰 - 30% 할인 이벤트! 🎧**\n\n귀하의 음악 경험을 완전히 변화시켜 줄 무선 블루투스 이어폰을 소개합니다! 원래 가격 100,000원이던 이 제품을 지금 만나보시면 **30% 할인된 가격인 70,000원**에 소유하실 수 있습니다. \n\n✅ **탁월한 음질**: 최첨단 오디오 기술이 적용되어 깊고 풍부한 사운드를 제공합니다. 음악을 더 가까이에서 느껴보세요!\n\n✅ **편리한 무선 디자인**: 끈이 없는 완전한 자유! 통화 및 음악 감상 중에도 편안하게 움직일 수 있습니다.\n\n✅ **오랜 배터리 수명**: 한번 충전으로 오랜 시간 동안 사용 가능하여, 하루 종일 귀하의 소중한 순간들과 함께할 수 있습니다.\n\n✅ **간편한 연결**: Bluetooth 5.0 기술로 빠르고 안정적인 연결을 경험해보세요. 스마트폰, 태블릿 등 다양한 기기와 호환됩니다.\n\n지금 회원 가입을 하시면 추가 할인 및 특별 혜택을 받을 수 있는 기회가 주어집니다. 끊임없이 변화하는 음악과 함께할 이 기회를 놓치지 마세요!\n\n💥 **지금 바로 구매하시면 할인이 적용된 가격으로 만나보실 수 있습니다!** 💥\n\n고객님, 이 특별한 제안이 절대적으로 한정된 기회임을 잊지 마세요. 음악의 새로운 세계로 저와 함께 뛰어들 준비가 되셨나요? 지금 바로 체크아웃하세요! 🎶✨', '한줄요약': '음악을 자유롭게 즐기세요, 특가 30% 할인된 무선 블루투스 이어폰으로 새로운 음향 경험을 만나보세요!', '알림톡': '안녕하세요, 고객님! 🎧 무선 블루투스 이어폰이 정가 100,000원에서 30% 할인된 70,000원에 판

{'product_name': '무선 블루투스 이어폰',
 'price': 100000,
 'is_on_sale': True,
 'discount_rate': 30,
 'current_time': '2026년 03월 23일 16시 14분',
 'user_name': None,
 '상세설명': '**🎧 무선 블루투스 이어폰 - 30% 할인 이벤트! 🎧**\n\n귀하의 음악 경험을 완전히 변화시켜 줄 무선 블루투스 이어폰을 소개합니다! 원래 가격 100,000원이던 이 제품을 지금 만나보시면 **30% 할인된 가격인 70,000원**에 소유하실 수 있습니다. \n\n✅ **탁월한 음질**: 최첨단 오디오 기술이 적용되어 깊고 풍부한 사운드를 제공합니다. 음악을 더 가까이에서 느껴보세요!\n\n✅ **편리한 무선 디자인**: 끈이 없는 완전한 자유! 통화 및 음악 감상 중에도 편안하게 움직일 수 있습니다.\n\n✅ **오랜 배터리 수명**: 한번 충전으로 오랜 시간 동안 사용 가능하여, 하루 종일 귀하의 소중한 순간들과 함께할 수 있습니다.\n\n✅ **간편한 연결**: Bluetooth 5.0 기술로 빠르고 안정적인 연결을 경험해보세요. 스마트폰, 태블릿 등 다양한 기기와 호환됩니다.\n\n지금 회원 가입을 하시면 추가 할인 및 특별 혜택을 받을 수 있는 기회가 주어집니다. 끊임없이 변화하는 음악과 함께할 이 기회를 놓치지 마세요!\n\n💥 **지금 바로 구매하시면 할인이 적용된 가격으로 만나보실 수 있습니다!** 💥\n\n고객님, 이 특별한 제안이 절대적으로 한정된 기회임을 잊지 마세요. 음악의 새로운 세계로 저와 함께 뛰어들 준비가 되셨나요? 지금 바로 체크아웃하세요! 🎶✨',
 '한줄요약': '음악을 자유롭게 즐기세요, 특가 30% 할인된 무선 블루투스 이어폰으로 새로운 음향 경험을 만나보세요!',
 '알림톡': '안녕하세요, 고객님! 🎧 무선 블루투스 이어폰이 정가 100,000원에서 30% 할인된 70,000원에 판매 중입니다! 🎉 지금 바로 확인해보세요

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

| Runnable | 역할 | 주요 사용법 |
|----------|------|-------------|
| `RunnablePassthrough` | 입력을 그대로 통과시켜요 | 분기점에서 원본 데이터 유지 |
| `RunnablePassthrough.assign()` | 원본 유지 + 새 키 추가 | 파생 값(할인율, 타임스탬프) 동적 주입 |
| `RunnableParallel` | 여러 체인 동시 실행 | 같은 입력으로 다양한 출력 생성 |
| `RunnableLambda` | Python 함수를 Runnable로 변환 | 전처리·후처리·데이터 타입 변환 |

세 가지를 조합하면 `enrich → parallel → merge` 패턴으로 복잡한 실무 파이프라인을 선언적으로 표현할 수 있어요.

## 다음 노트북 예고

다음 노트북에서는 **메모리(Memory)**와 **대화 이력 관리**를 다뤄요. 지금까지 수동으로 관리하던 대화 히스토리를 LangChain 컴포넌트가 자동으로 처리하는 방법을 알아볼게요.